In [1]:
# Data preprocessing in python for machine learning

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [3]:
planes_df = pd.read_csv('Data/planes_crashes.csv')
planes_df.head()

,Date,Time,Location,Operator,Flight #,Route,Type,Registration,cn/In,Aboard,Fatalities,Ground,Summary
0,09/17/1908,17:18,"Fort Myer, Virginia",Military - U.S. Army,NaN,Demonstration,Wright Flyer III,NaN,1,2.0,1.0,0.0,"During a demonstration flight, a U.S. Army fly..."
1,07/12/1912,06:30,"AtlantiCity, New Jersey",Military - U.S. Navy,NaN,Test flight,Dirigible,NaN,NaN,5.0,5.0,0.0,First U.S. dirigible Akron exploded just offsh...
2,08/06/1913,NaN,"Victoria, British Columbia, Canada",Private,-,NaN,Curtiss seaplane,NaN,NaN,1.0,1.0,0.0,The first fatal airplane accident in Canada oc...
3,09/09/1913,18:30,Over the North Sea,Military - German Navy,NaN,NaN,Zeppelin L-1 (airship),NaN,NaN,20.0,14.0,0.0,The airship flew into a thunderstorm and encou...
4,10/17/1913,10:30,"Near Johannisthal, Germany",Military - German Navy,NaN,NaN,Zeppelin L-2 (airship),NaN,NaN,30.0,30.0,0.0,Hydrogen gas which was being vented was sucked...


In [4]:
print(planes_df.isna().sum())

Date               0
Time            2219
Location          20
Operator          18
Flight #        4199
Route           1707
Type              27
Registration     335
cn/In           1228
Aboard            22
Fatalities        12
Ground            22
Summary          390
dtype: int64


In [5]:
print(planes_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5268 entries, 0 to 5267
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          5268 non-null   object 
 1   Time          3049 non-null   object 
 2   Location      5248 non-null   object 
 3   Operator      5250 non-null   object 
 4   Flight #      1069 non-null   object 
 5   Route         3561 non-null   object 
 6   Type          5241 non-null   object 
 7   Registration  4933 non-null   object 
 8   cn/In         4040 non-null   object 
 9   Aboard        5246 non-null   float64
 10  Fatalities    5256 non-null   float64
 11  Ground        5246 non-null   float64
 12  Summary       4878 non-null   object 
dtypes: float64(3), object(10)
memory usage: 535.2+ KB
None


In [6]:
planes_df["Date"] = pd.to_datetime(planes_df["Date"])

In [7]:
# Drop the cn/In and Route columns from planes_df (less useful for most analysis)
planes_cols = planes_df.drop(["cn/In", "Route"], axis=1)

# Drop rows with missing Fatalities values from planes_cols
planes_subset = planes_cols.dropna(subset=["Fatalities"])

# Print out the shape of the subset
print(planes_subset.shape)

(5256, 11)


In [8]:
print(planes_subset.isna().sum())

Date               0
Time            2207
Location          19
Operator          18
Flight #        4187
Type              25
Registration     334
Aboard            10
Fatalities         0
Ground            10
Summary          385
dtype: int64


In [9]:
planes_df['Operator'].dropna()

0                  Military - U.S. Army
1                  Military - U.S. Navy
2                               Private
3                Military - German Navy
4                Military - German Navy
                     ...               
5263    Military - Indonesian Air Force
5264                        Service Air
5265                         Air France
5266                         Strait Air
5267        Military - Indian Air Force
Name: Operator, Length: 5250, dtype: object

In [10]:
# Drop rows with missing Operator (can't train/predict a missing label)
planes_df = planes_df.dropna(subset=['Operator'])

# Keep only operators with at least 2 occurrences (required for stratified split)
operator_counts = planes_df['Operator'].value_counts()
valid_operators = operator_counts[operator_counts >= 2].index
planes_df = planes_df[planes_df['Operator'].isin(valid_operators)]

# Create a DataFrame with all columns except Operator
X = planes_df.drop("Operator", axis=1)

# Create an Operator labels dataset
y = planes_df[['Operator']]

# Use stratified sampling to split up the dataset according to the y dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Print the Operator counts from y_train
print(y_train['Operator'].value_counts())

Operator
Aeroflot                             134
Military - U.S. Air Force            132
Air France                            52
Deutsche Lufthansa                    49
Air Taxi                              33
                                    ... 
Saturn Airways                         1
Salair                                 1
Crossair                               1
Channel Flying Service - Air Taxi      1
Military - Colombian Army              1
Name: count, Length: 555, dtype: int64


In [11]:
# Sample data
x = np.array([1, 2, 3, 4, 5])
y = np.array([2, 4, 5, 4, 5])

# Add constant for intercept term
X = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, X).fit()

# Print summary
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.600
Model:                            OLS   Adj. R-squared:                  0.467
Method:                 Least Squares   F-statistic:                     4.500
Date:                Wed, 29 Jul 2026   Prob (F-statistic):              0.124
Time:                        06:34:21   Log-Likelihood:                -5.2598
No. Observations:                   5   AIC:                             14.52
Df Residuals:                       3   BIC:                             13.74
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.2000      0.938      2.345      0.1

In [12]:
# Load the wine dataset
wine = load_wine()
X = wine.data
y = wine.target

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

knn = KNeighborsClassifier()

# Fit the knn model to the training data
knn.fit(X_train, y_train)

# Score the model on the test data
print(knn.score(X_test, y_test))

0.7777777777777778


We see that without normalization, our accuracy score for the model is very low so we have to standardize it.

In [13]:
wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
print(wine_df.columns.tolist())

['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


In [14]:
print(wine_df.var())

alcohol                             0.659062
malic_acid                          1.248015
ash                                 0.075265
alcalinity_of_ash                  11.152686
magnesium                         203.989335
total_phenols                       0.391690
flavanoids                          0.997719
nonflavanoid_phenols                0.015489
proanthocyanins                     0.327595
color_intensity                     5.374449
hue                                 0.052245
od280/od315_of_diluted_wines        0.504086
proline                         99166.717355
dtype: float64


In [15]:
# Print out the variance of the Proline column
print(wine_df['proline'].var())

# Apply the log normalization function to the Proline column
wine_df['Proline_log'] = np.log(wine_df['proline'])

# Check the variance of the normalized Proline column
print(wine_df['Proline_log'].var())

99166.71735542428
0.17231366191842018


In [16]:
# Create the scaler
scaler = StandardScaler()

# Subset the DataFrame you want to scale 
wine_subset = wine_df[['ash', 'alcalinity_of_ash', 'magnesium']]

# Apply the scaler to wine_subset
wine_subset_scaled = scaler.fit_transform(wine_subset)

In [17]:
# Scale the training and test features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the k-nearest neighbors model to the training data
knn.fit(X_train_scaled, y_train)

# Score the model on the test data
print(knn.score(X_test_scaled, y_test))

0.9333333333333333


In [18]:
churn_df = pd.read_csv('Data/telecom_churn.csv')
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
print(churn_df['PhoneService'].nunique()) # This feature needs to be encoded 

2


In [20]:
# Set up the LabelEncoder object
enc = LabelEncoder()

# Apply the encoding to the "PhoneService" column
churn_df["PhoneService_enc"] = enc.fit_transform(churn_df["PhoneService"])

# Compare the two columns
print(churn_df[["PhoneService", "PhoneService_enc"]].head())

  PhoneService  PhoneService_enc
0           No                 0
1          Yes                 1
2          Yes                 1
3           No                 0
4          Yes                 1


In [21]:
print(churn_df['Churn'].unique()) # This label needs encoding

['No' 'Yes']


In [22]:
demo_df = pd.get_dummies(churn_df['Churn'], dtype=int)
demo_df.head(5)

,No,Yes
0,1,0
1,1,0
2,0,1
3,1,0
4,0,1


In [24]:
# Apply the encoding to the "Churn" column
churn_df["Churn_enc"] = enc.fit_transform(churn_df["Churn"])

# Compare the two columns
print(churn_df[["Churn", "Churn_enc"]].head())

  Churn  Churn_enc
0    No          0
1    No          0
2   Yes          1
3    No          0
4   Yes          1
